In [1]:
import os
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

ANALYSIS_DATE = pd.Timestamp("2026-06-03")  # fixed date for consistency

RAW_PATH = "D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/customer_purchase_data.csv"

CLEAN_PATH = "D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/customer_purchase_data_cleaned.csv"
ENRICHED_PATH = "D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/customer_purchase_data_enriched.csv"

OUT_CUSTOMER_KPIS = "D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/customer_kpis.csv"
OUT_MONTHLY_KPIS = "D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/monthly_kpis.csv"
OUT_CATEGORY_KPIS = "D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/category_kpis.csv"
OUT_SEGMENT_KPIS = "D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/segment_kpis.csv"
OUT_REGION_KPIS = "D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/region_kpis.csv"
OUT_PAYMENT_KPIS = "D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/payment_kpis.csv"
OUT_CHANNEL_KPIS = "D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/channel_kpis.csv"
OUT_LOYALTY_KPIS = "D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/loyalty_kpis.csv"

os.makedirs("D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data", exist_ok=True)

# -----------------------------
# Load
# -----------------------------
df = pd.read_csv(RAW_PATH)
print("Raw shape:", df.shape)
display(df.head())

# -----------------------------
# Required columns check
# -----------------------------
required = [
    "Customer ID","Customer Name","Age","Gender","City","Region","Occupation",
    "Product Category","Product Name","Purchase Date","Quantity Purchased",
    "Unit Price","Total Purchase Value","Payment Method","Purchase Channel",
    "Customer Segment","Loyalty Status","Discount Used","Purchase Frequency","Last Purchase Date"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError("Missing required columns: " + ", ".join(missing))

# -----------------------------
# Strip column names
# -----------------------------
df.columns = [c.strip() for c in df.columns]

# -----------------------------
# Standardize text fields
# -----------------------------
text_cols = [
    "Customer ID","Customer Name","Gender","City","Region","Occupation","Product Category",
    "Product Name","Payment Method","Purchase Channel","Customer Segment","Loyalty Status",
    "Discount Used","Purchase Frequency"
]
for c in text_cols:
    df[c] = df[c].astype(str).str.strip()

# Convert "nan"/"None" strings to real NaN
for c in text_cols:
    df[c] = df[c].replace({"nan": np.nan, "Nan": np.nan, "None": np.nan, "": np.nan})

# Customer name
df["Customer Name"] = df["Customer Name"].fillna("Unknown").str.title()

# Gender normalization
df["Gender"] = df["Gender"].replace({"M":"Male","F":"Female","male":"Male","female":"Female"})
df["Gender"] = df["Gender"].fillna("Unknown")

# Discount Used normalization
df["Discount Used"] = df["Discount Used"].replace({"Y":"Yes","N":"No","yes":"Yes","no":"No","TRUE":"Yes","FALSE":"No"})
df["Discount Used"] = df["Discount Used"].fillna("No")

# Payment normalization
df["Payment Method"] = df["Payment Method"].replace({"UPI":"UPI/Wallet","Wallet":"UPI/Wallet"})
df["Payment Method"] = df["Payment Method"].fillna("Unknown")

# Channel normalization
df["Purchase Channel"] = df["Purchase Channel"].replace({"App":"Mobile App","Web":"Website"})
df["Purchase Channel"] = df["Purchase Channel"].fillna("Unknown")

# Loyalty normalization
df["Loyalty Status"] = df["Loyalty Status"].replace({"NA":"None","N/A":"None"})
df["Loyalty Status"] = df["Loyalty Status"].fillna("None")

# Region normalization
df["Region"] = df["Region"].replace({"North East":"Northeast","Mid West":"Midwest"})
df["Region"] = df["Region"].fillna("Unknown")

# -----------------------------
# Types
# -----------------------------
df["Purchase Date"] = pd.to_datetime(df["Purchase Date"], errors="coerce")
df["Last Purchase Date"] = pd.to_datetime(df["Last Purchase Date"], errors="coerce")

num_cols = ["Age","Quantity Purchased","Unit Price","Total Purchase Value"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# -----------------------------
# Missing value handling
# -----------------------------
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Quantity Purchased"] = df["Quantity Purchased"].fillna(1)

# Unit Price: fill by category median, then global median
global_price = df["Unit Price"].median()
df["Unit Price"] = df.groupby("Product Category")["Unit Price"].transform(lambda s: s.fillna(s.median()))
df["Unit Price"] = df["Unit Price"].fillna(global_price)

# If Last Purchase Date missing -> use Purchase Date
df["Last Purchase Date"] = df["Last Purchase Date"].fillna(df["Purchase Date"])

# Drop rows where Purchase Date is missing (cannot analyze)
before_drop = df.shape[0]
df = df.dropna(subset=["Purchase Date"])
print("Dropped rows with missing Purchase Date:", before_drop - df.shape[0])

# -----------------------------
# Remove duplicates
# -----------------------------
before = df.shape[0]
df = df.drop_duplicates(subset=["Customer ID","Product Name","Purchase Date","Quantity Purchased","Unit Price"])
print("Duplicates removed:", before - df.shape[0])

# -----------------------------
# Validate totals
# -----------------------------
df["Calculated Total"] = (df["Quantity Purchased"] * df["Unit Price"]).round(2)
df["Total Mismatch Flag"] = np.where(df["Total Purchase Value"].round(2) != df["Calculated Total"], 1, 0)
mismatch_cnt = int(df["Total Mismatch Flag"].sum())

# Replace Total Purchase Value with calculated value (standard approach)
df["Total Purchase Value"] = df["Calculated Total"]
df.drop(columns=["Calculated Total"], inplace=True)

print("Total mismatches found (fixed by recalculation):", mismatch_cnt)

# -----------------------------
# Outlier flag (simple)
# -----------------------------
df["Outlier Flag"] = 0
df.loc[(df["Age"] < 16) | (df["Age"] > 90), "Outlier Flag"] = 1
q99_qty = df["Quantity Purchased"].quantile(0.99)
p99_price = df["Unit Price"].quantile(0.99)
df.loc[df["Quantity Purchased"] > q99_qty, "Outlier Flag"] = 1
df.loc[df["Unit Price"] > p99_price, "Outlier Flag"] = 1

# -----------------------------
# Feature engineering
# -----------------------------
df["Order Month"] = df["Purchase Date"].dt.to_period("M").astype(str)
df["Order Quarter"] = df["Purchase Date"].dt.to_period("Q").astype(str)
df["Order Weekday"] = df["Purchase Date"].dt.day_name()

bins = [0,24,34,44,54,150]
labels = ["18-24","25-34","35-44","45-54","55+"]
df["Age Group"] = pd.cut(df["Age"], bins=bins, labels=labels, right=True)

df["Discount Flag"] = np.where(df["Discount Used"].str.upper() == "YES", 1, 0)
df["Days Since Last Purchase"] = (ANALYSIS_DATE - df["Last Purchase Date"]).dt.days

# -----------------------------
# Save cleaned + enriched
# -----------------------------
clean_cols = required + ["Total Mismatch Flag","Outlier Flag"]
df_clean = df[clean_cols].copy()

df_clean.to_csv(CLEAN_PATH, index=False)
df.to_csv(ENRICHED_PATH, index=False)

print("Saved:", CLEAN_PATH)
print("Saved:", ENRICHED_PATH)

# -----------------------------
# KPI tables for Power BI
# -----------------------------
total_revenue = df["Total Purchase Value"].sum()
total_orders = df.shape[0]
total_customers = df["Customer ID"].nunique()
aov = total_revenue / total_orders if total_orders else 0

orders_per_customer = df.groupby("Customer ID")["Purchase Date"].count()
repeat_customers = int((orders_per_customer >= 2).sum())
repeat_rate = repeat_customers / total_customers if total_customers else 0

avg_orders_per_customer = total_orders / total_customers if total_customers else 0
basic_clv_12 = aov * avg_orders_per_customer * 12  # assumption

print("\n--- KPI SUMMARY ---")
print(f"Total Revenue: {total_revenue:,.2f}")
print(f"Total Orders: {total_orders:,}")
print(f"Total Customers: {total_customers:,}")
print(f"AOV: {aov:,.2f}")
print(f"Repeat Purchase Rate: {repeat_rate:.2%}")
print(f"Basic CLV (12 mo): {basic_clv_12:,.2f}")

# Customer KPIs
customer_kpis = (df
    .groupby(["Customer ID","Customer Name","Gender","Age Group","City","Region","Occupation","Customer Segment","Loyalty Status"], as_index=False)
    .agg(
        Total_Revenue=("Total Purchase Value","sum"),
        Orders=("Purchase Date","count"),
        Total_Quantity=("Quantity Purchased","sum"),
        Last_Purchase=("Last Purchase Date","max"),
        Discount_Usage_Rate=("Discount Flag","mean")
    )
)
customer_kpis["AOV"] = customer_kpis["Total_Revenue"] / customer_kpis["Orders"]
customer_kpis["Days Since Last Purchase"] = (ANALYSIS_DATE - customer_kpis["Last_Purchase"]).dt.days
customer_kpis.to_csv(OUT_CUSTOMER_KPIS, index=False)

# Monthly KPIs
monthly_kpis = (df.groupby("Order Month", as_index=False)
    .agg(Revenue=("Total Purchase Value","sum"), Orders=("Purchase Date","count"), Customers=("Customer ID","nunique"))
)
monthly_kpis["AOV"] = monthly_kpis["Revenue"] / monthly_kpis["Orders"]
monthly_kpis.to_csv(OUT_MONTHLY_KPIS, index=False)

# Category KPIs
category_kpis = (df.groupby("Product Category", as_index=False)
    .agg(Revenue=("Total Purchase Value","sum"), Orders=("Purchase Date","count"), Quantity=("Quantity Purchased","sum"))
)
category_kpis["AOV"] = category_kpis["Revenue"] / category_kpis["Orders"]
category_kpis.to_csv(OUT_CATEGORY_KPIS, index=False)

# Segment KPIs
segment_kpis = (df.groupby("Customer Segment", as_index=False)
    .agg(Revenue=("Total Purchase Value","sum"), Orders=("Purchase Date","count"), Customers=("Customer ID","nunique"))
)
segment_kpis["AOV"] = segment_kpis["Revenue"] / segment_kpis["Orders"]
segment_kpis.to_csv(OUT_SEGMENT_KPIS, index=False)

# Region KPIs
region_kpis = (df.groupby("Region", as_index=False)
    .agg(Revenue=("Total Purchase Value","sum"), Orders=("Purchase Date","count"), Customers=("Customer ID","nunique"))
)
region_kpis["AOV"] = region_kpis["Revenue"] / region_kpis["Orders"]
region_kpis.to_csv(OUT_REGION_KPIS, index=False)

# Payment KPIs
payment_kpis = (df.groupby("Payment Method", as_index=False)
    .agg(Revenue=("Total Purchase Value","sum"), Orders=("Purchase Date","count"))
)
payment_kpis["AOV"] = payment_kpis["Revenue"] / payment_kpis["Orders"]
payment_kpis.to_csv(OUT_PAYMENT_KPIS, index=False)

# Channel KPIs
channel_kpis = (df.groupby("Purchase Channel", as_index=False)
    .agg(Revenue=("Total Purchase Value","sum"), Orders=("Purchase Date","count"))
)
channel_kpis["AOV"] = channel_kpis["Revenue"] / channel_kpis["Orders"]
channel_kpis.to_csv(OUT_CHANNEL_KPIS, index=False)

# Loyalty KPIs
loyalty_kpis = (df.groupby("Loyalty Status", as_index=False)
    .agg(Revenue=("Total Purchase Value","sum"), Orders=("Purchase Date","count"), Customers=("Customer ID","nunique"))
)
loyalty_kpis["AOV"] = loyalty_kpis["Revenue"] / loyalty_kpis["Orders"]
loyalty_kpis.to_csv(OUT_LOYALTY_KPIS, index=False)

print("\nSaved KPI tables:")
print(OUT_CUSTOMER_KPIS)
print(OUT_MONTHLY_KPIS)
print(OUT_CATEGORY_KPIS)
print(OUT_SEGMENT_KPIS)
print(OUT_REGION_KPIS)
print(OUT_PAYMENT_KPIS)
print(OUT_CHANNEL_KPIS)
print(OUT_LOYALTY_KPIS)

Raw shape: (508, 20)


,Customer ID,Customer Name,Age,Gender,City,Region,Occupation,Product Category,Product Name,Purchase Date,Quantity Purchased,Unit Price,Total Purchase Value,Payment Method,Purchase Channel,Customer Segment,Loyalty Status,Discount Used,Purchase Frequency,Last Purchase Date
0,CUST0130,Leah Sharma,19,Female,Detroit,Midwest,Director,Groceries,Basmati Rice 5lb,2025-12-29,3,18.09,54.27,Credit Card,Website,Budget Buyers,Silver,No,Medium,2026-01-02
1,CUST0175,Vivaan Moore,52,Female,San Diego,West,Data Analyst,Beverages,Premium Coffee Pods,2026-05-21,1,14.30,14.30,Cash,Store,High Value,Gold,No,High,2026-06-03
2,CUST0084,Muhammad Davis,49,Female,San Diego,West,Teacher,Snacks,Trail Mix,2026-03-02,3,5.39,16.17,UPI/Wallet,Store,High Value,Gold,No,High,2026-03-29
3,CUST0022,Noah Taylor,34,Male,Orlando,South,Data Scientist,Personal Care,Shampoo,2025-12-18,2,6.49,12.98,Credit Card,Mobile App,Young Professionals,Silver,Yes,Medium,2026-02-08
4,CUST0027,Benjamin Das,46,Female,Newark,Northeast,Nurse,Home Care,Paper Towels,2026-05-07,1,3.37,3.37,Net Banking,Mobile App,Family Shoppers,NaN,No,Medium,2026-06-03


Dropped rows with missing Purchase Date: 0
Duplicates removed: 8
Total mismatches found (fixed by recalculation): 6
Saved: D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/customer_purchase_data_cleaned.csv
Saved: D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/customer_purchase_data_enriched.csv

--- KPI SUMMARY ---
Total Revenue: 12,851.96
Total Orders: 500
Total Customers: 170
AOV: 25.70
Repeat Purchase Rate: 84.71%
Basic CLV (12 mo): 907.20


C:\Users\Debarati\AppData\Local\Temp\ipykernel_15424\4158923814.py:203: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["Customer ID","Customer Name","Gender","Age Group","City","Region","Occupation","Customer Segment","Loyalty Status"], as_index=False)


MemoryError: Unable to allocate 4.08 GiB for an array with shape (268600, 16320) and data type int8